In [25]:
import json
#import torch
from transformers import BertTokenizer, TFBertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tqdm import tqdm

In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
import os
data_dir = '/content/drive/MyDrive/Healthcare_Sentiment/NewsAPI/train_items.jl'  # Replace with your actual path

In [ ]:
# Load data from JSON Lines file
#def load_data(file_path):
#    texts = []
#    labels = []
#    with open(file_path, 'r') as f:
#        for line in f:
#            st = r'%s' % line
#            data = json.loads(st)
#            texts.append(data['text'])
#            labels.append(data['label'])
#    return texts, labels

In [28]:
def load_data_with_cleaning(file_path):
        texts = []
        labels = []
        with open(file_path, 'r') as f:
            for line in f:
                try:
                    # Example cleaning: Replace literal backslashes with escaped ones
                    # This is a simplification and might not work for all issues!
                    cleaned_line = line.replace('\\"', '/')
                    #cleaned_line = cleaned_line1.replace('','')
                    # Ensure the cleaned line is a valid JSON string
                    data = json.loads(cleaned_line)
                    #combined_text = f"{data['title']} {data['text']}"
                    texts.append(data['title'])
                    #labels.append(data['label'])
                    # Convert label to integer
                    labels.append(int(data['label']))
                except json.JSONDecodeError as e:
                    print(f"Error decoding JSON on line: {e}")
                    print(f"Problematic line content (first 200 chars): {line[:1535]}")
                    # Decide how to handle errors (skip line, raise error, etc.)
                    continue # Skip the problematic line

        return texts, labels

In [51]:
texts, labels = load_data_with_cleaning(data_dir)

In [52]:
print(texts)

['Injectable Male Birth Control Effective for at Least 2 Years, Says Biotech Startup', "18 high-paying healthcare jobs that don't need a bachelor's degree", 'Semaglutide Shows Major Promise for Treating Serious Liver Disease', 'To See Within: Detecting X-Rays', 'Breaking the ‘intellectual bottleneck’: How AI is computing the previously uncomputible in healthcare', 'Immunotherapy drug capable of eliminating tumors in some early-stage cancers: Study', 'America’s Science Agency Says It Will Cut Funding to Researchers Who Protest Israel', "Newcastle's Howe back at work after hospital stay", "Howe is 'OK' but 'not 100%' after hospital stay", "Here's an exclusive look at the pitch deck that got an ex-Amazon exec $10 million to bring AI agents to health systems", 'ICE provides details on arrest that sparked protest at Rhode Island Hospital. What we know', 'Injectable Male Birth Control Effective for at Least 2 Years, Says Biotech Startup', "18 high-paying healthcare jobs that don't need a bac

In [ ]:
#load_data(data_dir)

In [53]:
#Split data
train_texts, val_texts, train_labels, val_labels = train_test_split(texts, labels, test_size=0.2, random_state=42)


In [54]:
print(f"Number of training samples: {len(train_texts)}")

Number of training samples: 56


In [55]:
# Load tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = TFBertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(set(labels)))

All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [56]:
# Tokenize data
train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=100)
val_encodings = tokenizer(list(val_texts), truncation=True, padding=True, max_length=100)

In [57]:
# Convert to TensorFlow datasets
train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_encodings),
    train_labels
)).batch(4)

In [58]:
val_dataset = tf.data.Dataset.from_tensor_slices((
    dict(val_encodings),
    val_labels
)).batch(4)

In [59]:
# Optimizer and loss
optimizer = tf.keras.optimizers.Adam(learning_rate=5e-5)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metric = tf.keras.metrics.SparseCategoricalAccuracy('accuracy')

In [60]:
# Compile model
model.compile(optimizer=optimizer, loss=loss, metrics=[metric])

In [61]:
# Train model
model.fit(train_dataset, epochs=5, validation_data=val_dataset
    )

Epoch 1/5
14/14 [==============================] - 48s 470ms/step - loss: 0.6838 - accuracy: 0.5714 - val_loss: 0.6460 - val_accuracy: 0.5714
Epoch 2/5
14/14 [==============================] - 1s 82ms/step - loss: 0.5777 - accuracy: 0.7321 - val_loss: 0.4744 - val_accuracy: 0.8571
Epoch 3/5
14/14 [==============================] - 1s 77ms/step - loss: 0.3150 - accuracy: 0.9464 - val_loss: 0.2140 - val_accuracy: 0.9286
Epoch 4/5
14/14 [==============================] - 1s 77ms/step - loss: 0.0471 - accuracy: 1.0000 - val_loss: 0.3540 - val_accuracy: 0.8571
Epoch 5/5
14/14 [==============================] - 1s 78ms/step - loss: 0.0085 - accuracy: 1.0000 - val_loss: 0.0775 - val_accuracy: 0.9286


In [62]:
# Evaluate model
loss, accuracy = model.evaluate(val_dataset)
print(f"Loss: {loss}, Accuracy: {accuracy}")

4/4 [==============================] - 0s 27ms/step - loss: 0.0775 - accuracy: 0.9286
Loss: 0.07751432806253433, Accuracy: 0.9285714030265808


In [63]:
# Make predictions
text = "This movie was great!"
predict_input = tokenizer(text, truncation=True, padding=True, return_tensors='tf')
output = model(predict_input)[0]
prediction_value = tf.argmax(output, axis=1).numpy()[0]
print(f"Predicted sentiment: {prediction_value}")

Predicted sentiment: 1


In [64]:
#File to Apply model to
new_data_dir = '/content/drive/MyDrive/Healthcare_Sentiment/NewsAPI/items.jl'  # Replace with your actual path

In [65]:
# Function to load data from a JSON Lines file (similar to your existing function)
def load_new_data_with_cleaning(file_path):
    contents = []
    texts = []
    ids = [] # Assuming each item has a unique ID you want to keep

    source = []
    descriptions = []
    author = []
    timestamp = []
    urlImage = []
    tags = []

    with open(file_path, 'r') as f:
        for line in f:
            try:
                cleaned_line = line.replace('\\"', '/')
                data = json.loads(cleaned_line)
                # Assuming you want to predict on the 'title' again
                texts.append(data['title'])
                contents.append(data['content'])


                source.append(data['source'])
                descriptions.append(data['description'])
                author.append(data['author'])
                timestamp.append(data['publishedAt'])
                urlImage.append(data['urlToImage'])
                tags.append(data['tags'])


                # Assuming an 'id' field exists to track the original item
                if 'url' in data:
                  ids.append(data['url'])
                else:
                  ids.append(None) # Or handle cases without an ID
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON on line: {e}")
                print(f"Problematic line content (first 200 chars): {line[:1535]}")
                continue
    return contents, texts, ids, source, descriptions, author, timestamp, urlImage, tags

# Load the new data
new_content, new_texts, new_ids, new_source, new_description, new_author, new_timestamp, new_urlToImage, new_tags = load_new_data_with_cleaning(new_data_dir)

# Tokenize the new data
new_encodings = tokenizer(list(new_texts), truncation=True, padding=True, max_length=100, return_tensors='tf')

# Create a TensorFlow dataset for the new data
new_dataset = tf.data.Dataset.from_tensor_slices(
    dict(new_encodings)
).batch(4)

# Make predictions
predictions = model.predict(new_dataset)

# Get the predicted class (index with the highest probability)
predicted_labels = tf.argmax(predictions.logits, axis=1).numpy()

# Write the results back to a new file or overwrite the original
output_data_dir = '/content/drive/MyDrive/Healthcare_Sentiment/NewsAPI/new_items_with_predictions.jl' # Define output file path

with open(output_data_dir, 'w') as outfile:
    for i, text in enumerate(new_texts):
        result = {
            'source': new_source[i],
            'author': new_author[i],
            'title': new_texts[i],
            'description': new_description[i],
            'url': new_ids[i], # Include the original ID if available
            'urlToImage': new_urlToImage[i],
            'publishedAt': new_timestamp[i],
            'content': new_content[i],
            'tags': new_tags[i],
            'predicted_label': int(predicted_labels[i]) # Ensure it's an integer
        }
        outfile.write(json.dumps(result) + '\n')

print(f"Predictions written to {output_data_dir}")

246/246 [==============================] - 11s 35ms/step
Predictions written to /content/drive/MyDrive/Healthcare_Sentiment/NewsAPI/new_items_with_predictions.jl
